In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [3]:
df = pd.read_csv("/Users/saif/Desktop/ML_Load_Forecasting/data/Gold/df_features_fourier_time_encoding.csv", index_col=0 , parse_dates=True)

print(df.head(2))

                      Load_MW  temperature_2m  wind_speed_10m  \
2020-01-07 23:00:00  55818.41             3.0            15.5   
2020-01-08 00:00:00  54220.74             2.5            15.9   

                     shortwave_radiation  hour  dayofweek  month  is_weekend  \
2020-01-07 23:00:00                  0.0    23          1      1           0   
2020-01-08 00:00:00                  0.0     0          2      1           0   

                     hour_sin  hour_cos  ...  month_sin  month_cos  \
2020-01-07 23:00:00 -0.258819  0.965926  ...        0.5   0.866025   
2020-01-08 00:00:00  0.000000  1.000000  ...        0.5   0.866025   

                     is_holiday  lag_hour   lag_day  lag_week  \
2020-01-07 23:00:00           0  54220.74  51875.48  57004.20   
2020-01-08 00:00:00           0  52752.95  49243.74  53772.35   

                     rolling_mean_24h  rolling_mean_168h  rolling_std_24h  \
2020-01-07 23:00:00      64713.215833       55537.921071      8552.620952   
20

### Hypertuning for Random Forest and XGBoost

In [4]:
X = df.drop(columns=["Load_MW"])
y = df["Load_MW"]

X_train = X[X.index.year < 2023]
y_train = y[y.index.year < 2023]

X_test = X[X.index.year == 2023]
y_test = y[y.index.year == 2023]

#### Hypertuning for Random Forest

In [5]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [6]:

param_grid_rf = {
    "n_estimators":     [100, 200, 300, 500],
    "max_depth":        [10, 20, 30, None],
    "min_samples_leaf": [1, 2, 4],
    "max_features":     [0.5, 0.7, 1.0]
}

tscv = TimeSeriesSplit(n_splits=5)

clf = RandomizedSearchCV(
    estimator=RandomForestRegressor(random_state=42, n_jobs=-1),
    param_distributions = param_grid_rf,
    n_iter     = 20,     
    cv         = tscv,
    scoring    = "neg_mean_absolute_error",
    random_state = 42,
    n_jobs     = -1,
    verbose    = 1 

)

search = clf.fit(X_train,y_train)
search.best_params_
print("Beste Parameter RF:", search.best_params_)

Fitting 5 folds for each of 20 candidates, totalling 100 fits
Beste Parameter RF: {'n_estimators': 500, 'min_samples_leaf': 1, 'max_features': 0.7, 'max_depth': 20}


In [7]:
y_pred_rf_tuned = search.best_estimator_.predict(X_test)
mae_rf_tuned  = mean_absolute_error(y_test, y_pred_rf_tuned)
rmse_rf_tuned = np.sqrt(mean_squared_error(y_test, y_pred_rf_tuned))
r2_rf_tuned   = r2_score(y_test, y_pred_rf_tuned)

In [8]:
print(f"RF Tuned → MAE: {mae_rf_tuned:.0f} MW | RMSE: {rmse_rf_tuned:.0f} MW | R²: {r2_rf_tuned:.4f}")

RF Tuned → MAE: 560 MW | RMSE: 740 MW | R²: 0.9936


#### hypertuning for XGBoost

In [9]:
from xgboost import XGBRegressor

In [10]:
param_grid_xgb = {
    "n_estimators":     [100, 200, 300, 500],
    "learning_rate":    [0.01, 0.05, 0.1, 0.2],
    "max_depth":        [3, 5, 7, 10],
    "subsample":        [0.6, 0.8, 1.0],
    "colsample_bytree": [0.6, 0.8, 1.0]
}

xgb_search = RandomizedSearchCV(
    estimator = XGBRegressor(random_state=42, n_jobs=-1),
    param_distributions = param_grid_xgb,
    n_iter       = 20,
    cv           = tscv,
    scoring      = "neg_mean_absolute_error",
    random_state = 42,
    n_jobs       = -1,
    verbose      = 1
)

xgb_search.fit(X_train, y_train)
print("Beste Parameter XGB:", xgb_search.best_params_)

Fitting 5 folds for each of 20 candidates, totalling 100 fits
Beste Parameter XGB: {'subsample': 0.6, 'n_estimators': 300, 'max_depth': 5, 'learning_rate': 0.1, 'colsample_bytree': 1.0}


In [11]:
y_pred_xgb_tuned = xgb_search.best_estimator_.predict(X_test)
mae_xgb_tuned  = mean_absolute_error(y_test, y_pred_xgb_tuned)
rmse_xgb_tuned = np.sqrt(mean_squared_error(y_test, y_pred_xgb_tuned))
r2_xgb_tuned   = r2_score(y_test, y_pred_xgb_tuned)

In [12]:
print(f"XGB Tuned → MAE: {mae_xgb_tuned:.0f} MW | RMSE: {rmse_xgb_tuned:.0f} MW | R²: {r2_xgb_tuned:.4f}")

XGB Tuned → MAE: 520 MW | RMSE: 676 MW | R²: 0.9946


#### Save the Models

In [13]:
import joblib
joblib.dump(search.best_estimator_, "/Users/saif/Desktop/ML_Load_Forecasting/models/rf_tuned.pkl")
joblib.dump(xgb_search.best_estimator_, "/Users/saif/Desktop/ML_Load_Forecasting/models/xgb_tuned.pkl")

['/Users/saif/Desktop/ML_Load_Forecasting/models/xgb_tuned.pkl']